# CataLIST — EuMINe DataBridge Hackathon 2026
## Stage 2 Demo Notebook

**Team:** CataLIST (Luxembourg Institute of Science and Technology — LIST)  
**Model:** ALIGNN + MACE-MP-0 weighted ensemble with isotonic calibration  
**Version:** v4_combined (score: 38.38 / 40 on validation set)  
**Repository:** https://github.com/Iman-Peivaste/eumine_databridge_2026

---

### What this notebook demonstrates
1. **Setup** — install dependencies
2. **Model download** — pull artifacts from Hugging Face Hub (~33 MB)
3. **Load predictor** — MatFed API v1 compliant
4. **MatFed compliance** — all API checks
5. **Live prediction** — Si, NaCl, TiO₂ and real JARVIS structures
6. **Training overview** — how the model was trained (read-only)
7. **Federation demo** — `FederatedEnsemble` with calibration + prediction

> **Runtime:** Select *Runtime → Change runtime type → T4 GPU* for best performance (~8 min total)

---
## [1] Setup — Install Dependencies
One-time installation (~3 minutes on Colab T4).

In [ ]:
import subprocess, sys, os, importlib, site

# REPO is the clone of eumine_databridge_2026 — setup.py is at its root
REPO     = "/content/eumine_databridge_2026"
API_REPO = "/content/eumine_hackathon_2026"   # separate repo containing matfed_api

def run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"Failed: {cmd}")
    return r.stdout

def _add_matfed_to_path():
    matfed_dir = f"{API_REPO}/matfed-api-template"
    if not os.path.isdir(matfed_dir):
        return
    if matfed_dir not in sys.path:
        sys.path.insert(0, matfed_dir)
    pth = os.path.join(site.getsitepackages()[0], "matfed_api_colab.pth")
    if not os.path.exists(pth):
        with open(pth, "w") as f:
            f.write(matfed_dir + "\n")

def _packages_ready():
    # Check sklearn version: bg_head.joblib was pickled with 1.5.x;
    # sklearn 1.6+ renamed CyHalfSquaredError and breaks loading.
    try:
        import sklearn
        if not sklearn.__version__.startswith("1.5."):
            print(f"  sklearn {sklearn.__version__} detected — need 1.5.x, will reinstall")
            return False
    except ImportError:
        return False
    for pkg in ("eumine_databridge", "alignn", "dgl"):
        if importlib.util.find_spec(pkg) is None:
            return False
    return True

# ── After a runtime restart packages are on disk — just wire matfed path ──
if _packages_ready():
    _add_matfed_to_path()
    print("✓ All packages installed — run the cells below")
else:
    print("First-time setup (~3 min). Runtime will restart once at the end.\n")

    # 1. Clone repos
    if not os.path.exists(REPO):
        print("[1/5] Cloning main repo...")
        run(f"git clone --depth 1 https://github.com/Iman-Peivaste/eumine_databridge_2026.git {REPO}")
        print("      ✓ Done")
    else:
        print("[1/5] ✓ Main repo already present")

    if not os.path.exists(API_REPO):
        print("      Cloning MatFed API repo...")
        run(f"git clone --depth 1 https://github.com/EuMINe-COST/eumine_hackathon_2026.git {API_REPO}")
        print("      ✓ Done")
    else:
        print("      ✓ MatFed API repo already present")

    # 2. Install DGL — try CUDA-enabled wheel first, fall back to CPU-only
    import torch
    tv = torch.__version__.split("+")[0]
    mm = ".".join(tv.split(".")[:2])
    print(f"[2/5] Installing DGL for PyTorch {tv}...")

    # Build ordered list of wheel URLs to try (CUDA first)
    dgl_candidates = []
    if torch.cuda.is_available():
        cuda_ver = torch.version.cuda                      # e.g. "12.4"
        cuda_tag = "cu" + "".join(cuda_ver.split(".")[:2]) # "cu124"
        dgl_candidates += [
            f"pip install -q dgl -f https://data.dgl.ai/wheels/{cuda_tag}/torch-{mm}/repo.html",
            "pip install -q dgl -f https://data.dgl.ai/wheels/cu121/torch-{}/repo.html".format(mm),
        ]
    dgl_candidates += [
        f"pip install -q dgl -f https://data.dgl.ai/wheels/torch-{mm}/repo.html",
        "pip install -q dgl",
    ]

    # Test helper: import dgl AND verify .to(device) works if CUDA expected
    def _test_dgl(need_cuda):
        move = (
            "import torch; g.to(torch.device('cuda'))"
            if need_cuda else "pass"
        )
        return subprocess.run(
            f"python -c \""
            f"import sys, types; "
            f"sys.modules.setdefault('dgl.graphbolt', types.ModuleType('dgl.graphbolt')); "
            f"import dgl; g = dgl.graph(([0],[1])); {move}; print(dgl.__version__)\"",
            shell=True, capture_output=True, text=True)

    dgl_ok = False
    for dgl_cmd in dgl_candidates:
        run(dgl_cmd, check=False)
        t = _test_dgl(need_cuda=torch.cuda.is_available())
        if t.returncode == 0:
            print(f"      ✓ DGL {t.stdout.strip()}")
            dgl_ok = True
            break
    if not dgl_ok:
        # CPU-only DGL is fine — ALIGNNFineTuner falls back to CPU automatically
        print("      ⚠ CUDA DGL unavailable — ALIGNN will run on CPU (still works)")

    # 3. Core packages — sklearn pinned to 1.5.x to match pickled artifacts
    print("[3/5] Installing packages...")
    run("pip install -q 'scikit-learn==1.5.1' pymatgen alignn mace-torch torchdata "
        "'huggingface_hub>=0.20' 'optuna>=3.6' python-dotenv joblib")
    print("      ✓ Done")

    # 4. Local packages
    print("[4/5] Installing local packages...")
    run(f"pip install -q -e {REPO}")
    print(f"      ✓ eumine_databridge")
    _add_matfed_to_path()
    print(f"      ✓ matfed_api (via sys.path)")

    # 5. Restart runtime
    print("\n[5/5] Restarting runtime...")
    print("      After reconnecting, run ALL cells again — setup will be skipped.")
    import time; time.sleep(2)
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        os.kill(os.getpid(), 9)

---
## [2] Download Model Artifacts from Hugging Face Hub

Custom artifacts are ~33 MB. The MACE base model (~43 MB) downloads automatically on first inference.

| Artifact | Size |
|---|---|
| `alignn_ef_combined/best_model.pt` | 16 MB |
| `alignn_bg_combined/best_model.pt` | 16 MB |
| `mace_artifacts/` (BG head, references) | ~961 KB |
| `calibration/` (isotonic calibrators) | ~3.5 KB |
| `ensemble_weights.json` | ~215 B |

In [ ]:
from huggingface_hub import snapshot_download
import os

MODEL_PATH = "/content/catallist_v4_combined"

if not os.path.exists(f"{MODEL_PATH}/ensemble_weights.json"):
    print("Downloading model artifacts from Hugging Face Hub...")
    snapshot_download(
        repo_id="Imanpeivaste/catallist_v4_combined",
        repo_type="model",
        local_dir=MODEL_PATH,
    )
    print("  Download complete.")
else:
    print("  Model artifacts already present.")

# Verify essential files
required = [
    f"{MODEL_PATH}/alignn_ef_combined/best_model.pt",
    f"{MODEL_PATH}/alignn_bg_combined/best_model.pt",
    f"{MODEL_PATH}/ensemble_weights.json",
    f"{MODEL_PATH}/calibration/ef_calibrator.joblib",
    f"{MODEL_PATH}/mace_artifacts/ref_energies.json",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"✗ Missing files: {missing}")
else:
    sizes = {os.path.basename(f): f"{os.path.getsize(f)/1e6:.1f} MB" for f in required}
    for name, size in sizes.items():
        print(f"  ✓ {name}: {size}")
    print("\n✓ All artifacts present")

---
## [3] Load Predictor (MatFed API v1)

Loads ALIGNN EF, ALIGNN BG, MACE-MP-0, ensemble weights and calibration layer.

In [ ]:
import os, sys, types

# ── DGL graphbolt compatibility patch ─────────────────────────────────────
# Some Colab PyTorch versions include a libgraphbolt_pytorch_X.Y.Z.so that
# doesn't ship with certain DGL wheels.  Pre-stubbing dgl.graphbolt before
# the first `import dgl` makes Python skip loading that .so entirely.
# ALIGNN only uses basic DGL operations and never calls graphbolt APIs.
if "dgl" not in sys.modules and "dgl.graphbolt" not in sys.modules:
    _gb = types.ModuleType("dgl.graphbolt")
    _gb.load_graphbolt = lambda: None
    sys.modules["dgl.graphbolt"] = _gb

# ── Environment ───────────────────────────────────────────────────────────
REPO      = "/content/eumine_databridge_2026"
MODEL_PATH = "/content/catallist_v4_combined"

os.environ["MATFED_MODEL_PATH"] = MODEL_PATH
os.environ["WANDB_MODE"] = "disabled"

# ── Load predictor ────────────────────────────────────────────────────────
from eumine_databridge.matfed.predictor import LISTEuMINePredictor

predictor = LISTEuMINePredictor()

print("\n" + "="*60)
print("PREDICTOR DESCRIPTION")
print("="*60)
desc = predictor.describe()
for key, value in desc.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    elif isinstance(value, list):
        print(f"  {key}: {', '.join(str(v) for v in value)}")
    else:
        print(f"  {key}: {value}")
print("="*60)
print("\n✓ Predictor loaded")

---
## [4] MatFed API Compliance Checks

Verifies all required MatFed API v1 interface requirements.

In [ ]:
import numpy as np
from pymatgen.core import Structure, Lattice

results = {}

def check(label, fn):
    try:
        val = fn()
        print(f"  ✓ {label}: {val}")
        results[label] = True
    except Exception as e:
        print(f"  ✗ {label}: {e}")
        results[label] = False

# Build a test structure
si = Structure(Lattice.cubic(5.43), ["Si", "Si"], [[0,0,0],[0.25,0.25,0.25]])

print("MatFed API v1 compliance:")
print("-" * 50)

check("describe() returns dict",
      lambda: type(predictor.describe()).__name__)

check("team_name present",
      lambda: predictor.describe()["team_name"])

check("model_type present",
      lambda: predictor.describe()["model_type"][:40] + "...")

check("api_version present",
      lambda: predictor.describe()["api_version"])

check("data_sources present",
      lambda: len(predictor.describe()["data_sources"]))

check("uncertainty_available",
      lambda: predictor.describe()["uncertainty_available"])

check("predict() returns list",
      lambda: type(predictor.predict([si])).__name__)

check("formation_energy_per_atom is float",
      lambda: type(predictor.predict([si])[0]["formation_energy_per_atom"]).__name__)

check("band_gap is float >= 0",
      lambda: round(predictor.predict([si])[0]["band_gap"], 4))

check("uncertainty_ef present",
      lambda: round(predictor.predict([si])[0]["uncertainty_ef"], 4))

check("model_id present",
      lambda: predictor.predict([si])[0]["model_id"])

print("-" * 50)
passed = sum(results.values())
total = len(results)
status = "✓ ALL PASSED" if passed == total else f"✗ {total - passed} FAILED"
print(f"  RESULT: {passed}/{total} checks — {status}")

---
## [5] Live Prediction Demo

### 5a — Programmatic structures (Si, NaCl, TiO₂)
Classic benchmark crystals with known properties.

In [ ]:
from pymatgen.core import Structure, Lattice

structures_prog = []
labels_prog = []

# Silicon (diamond cubic) — known EF ~ -0.54 eV/atom, BG ~ 1.1 eV
si = Structure(Lattice.cubic(5.43), ["Si", "Si"], [[0,0,0],[0.25,0.25,0.25]])
structures_prog.append(si); labels_prog.append("Si (diamond)")

# NaCl (rock salt) — known EF ~ -3.37 eV/atom, BG ~ 8.5 eV (ionic)
nacl = Structure(Lattice.cubic(5.64), ["Na", "Cl"], [[0,0,0],[0.5,0.5,0.5]])
structures_prog.append(nacl); labels_prog.append("NaCl (rock salt)")

# TiO2 rutile — known EF ~ -3.24 eV/atom, BG ~ 3.0 eV
a, c = 4.594, 2.959
tio2 = Structure(
    Lattice.tetragonal(a, c),
    ["Ti", "O", "O"],
    [[0,0,0],[0.305,0.305,0],[0.695,0.695,0]]
)
structures_prog.append(tio2); labels_prog.append("TiO₂ (rutile)")

preds = predictor.predict(structures_prog)

print(f"{'Structure':<20} {'EF (eV/atom)':>14} {'BG (eV)':>10} {'±EF':>8} {'±BG':>8} {'Model':<30}")
print("-" * 92)
for label, p in zip(labels_prog, preds):
    print(f"{label:<20} {p['formation_energy_per_atom']:>14.4f} {p['band_gap']:>10.4f} "
          f"{p['uncertainty_ef']:>8.4f} {p['uncertainty_bg']:>8.4f} {p['model_id']:<30}")

### 5b — Real structures from the dataset (JARVIS augmented CIFs)

Loading CIF files directly from the GitHub repository (`data/augmented/structures/`).

In [ ]:
from pathlib import Path
from pymatgen.core import Structure

cif_dir = Path(REPO) / "eumine_databridge" / "data" / "augmented" / "structures"
cif_files = sorted(cif_dir.glob("*.cif"))[:5]

structures_cif = []
ids_cif = []
for cif in cif_files:
    try:
        s = Structure.from_file(str(cif))
        structures_cif.append(s)
        ids_cif.append(cif.stem)
    except Exception as e:
        print(f"  Skipping {cif.name}: {e}")

print(f"Loaded {len(structures_cif)} structures from {cif_dir.name}/\n")

preds_cif = predictor.predict(structures_cif)

print(f"{'Material ID':<20} {'Formula':<12} {'Sites':>5} {'EF (eV/atom)':>14} {'BG (eV)':>10} {'±EF':>8}")
print("-" * 75)
for mat_id, s, p in zip(ids_cif, structures_cif, preds_cif):
    print(f"{mat_id:<20} {s.composition.reduced_formula:<12} {len(s):>5} "
          f"{p['formation_energy_per_atom']:>14.4f} {p['band_gap']:>10.4f} {p['uncertainty_ef']:>8.4f}")

---
## [6] Training Overview (Reference)

> **This cell does not train anything.** It shows how the model was trained — the architecture, hyperparameters, and validation convergence curves.

The final model (`v4_combined`) was trained on the merged train+val Bridge Dataset split with augmented data from JARVIS (layered/2D materials and rare-earth compounds).

In [ ]:
# ── Training configuration (read-only display) ──────────────────────────────
import json

ef_config = {
    "target": "formation_energy_per_atom",
    "pretrained_from": "JARVIS-DFT ALIGNN (jv_formation_energy_peratom_alignn)",
    "alignn_layers": 4,
    "gcn_layers": 4,
    "hidden_features": 256,
    "cutoff_angstrom": 8.0,
    "max_neighbors": 12,
    "epochs": 300,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "scheduler": "CosineAnnealingLR",
    "loss": "HuberLoss (delta=0.5)",
    "freeze_encoder_epochs": 20,
    "early_stopping_patience": 50,
}
bg_config = {
    "target": "band_gap",
    "pretrained_from": "JARVIS-DFT ALIGNN (jv_optb88vdw_bandgap_alignn)",
    "alignn_layers": 6,
    "gcn_layers": 4,
    "hidden_features": 256,
    "cutoff_angstrom": 8.0,
    "max_neighbors": 12,
    "epochs": 400,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "scheduler": "CosineAnnealingLR",
    "loss": "HuberLoss (delta=1.0)",
    "freeze_encoder_epochs": 30,
    "early_stopping_patience": 60,
}

print("ALIGNN Formation Energy Config:")
for k, v in ef_config.items():
    print(f"  {k:<30}: {v}")

print("\nALIGNN Band Gap Config:")
for k, v in bg_config.items():
    print(f"  {k:<30}: {v}")

print("\nEnsemble:")
ew = json.loads(open(f"{MODEL_PATH}/ensemble_weights.json").read())
print(f"  EF weights — ALIGNN: {ew['weights_ef']['alignn']:.3f}, MACE: {ew['weights_ef']['mace']:.3f}")
print(f"  BG weights — ALIGNN: {ew['weights_bg']['alignn']:.3f}, MACE: {ew['weights_bg']['mace']:.3f}")
print(f"  Optimization: Optuna TPE sampler, 300 trials")
print(f"  Calibration:  isotonic regression (post-hoc)")

print("\nTraining command:")
print("  python scripts/retrain_combined_augmented.py")
print("  # Trains on train+val+augmented data for final submission")

In [ ]:
# ── Validation loss curve ────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
from pathlib import Path

ef_history_path = Path(MODEL_PATH) / "alignn_ef_combined" / "training_history.json"
bg_history_path = Path(MODEL_PATH) / "alignn_bg_combined" / "training_history.json"

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("ALIGNN Training — Validation MAE Convergence (v4_combined)", fontsize=13)

baselines = {"Formation Energy": 0.2378, "Band Gap": 0.6414}
best_maes = {"Formation Energy": None, "Band Gap": None}

for ax, path, title, ylabel, baseline in zip(
    axes,
    [ef_history_path, bg_history_path],
    ["Formation Energy", "Band Gap"],
    ["Val MAE (eV/atom)", "Val MAE (eV)"],
    [0.2378, 0.6414],
):
    if path.exists():
        h = json.loads(path.read_text())
        val_mae = h["val_mae"]
        epochs = list(range(1, len(val_mae) + 1))
        best_epoch = val_mae.index(min(val_mae)) + 1
        best_maes[title] = min(val_mae)

        ax.plot(epochs, val_mae, color="steelblue", linewidth=1.5, label="Val MAE")
        ax.axhline(baseline, color="red", linestyle="--", linewidth=1.2,
                   label=f"Baseline ({baseline})")
        ax.axvline(best_epoch, color="green", linestyle=":", linewidth=1.2,
                   label=f"Best epoch ({best_epoch})")
        ax.scatter([best_epoch], [min(val_mae)], color="green", s=60, zorder=5)

        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title}\nBest: {min(val_mae):.4f} (epoch {best_epoch})")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, "training_history.json not found",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)

plt.tight_layout()
plt.show()

print("\nFinal performance:")
from eumine_databridge.utils.metrics import compute_full_score
ef_mae = best_maes["Formation Energy"] or 0.0797
bg_mae = best_maes["Band Gap"] or 0.2346
score = compute_full_score(ef_mae, bg_mae)
print(f"  EF val MAE  : {ef_mae:.4f} eV/atom  (baseline: 0.2378)")
print(f"  BG val MAE  : {bg_mae:.4f} eV        (baseline: 0.6414)")
print(f"  EF score    : {score['score_ef']:.2f} / 20")
print(f"  BG score    : {score['score_bg']:.2f} / 20")
print(f"  TOTAL       : {score['total_performance_score']:.2f} / 40")
print(f"  Beats baseline: EF={score['beats_baseline_ef']}, BG={score['beats_baseline_bg']}")

---
## [7] Federation Demo

The `FederatedEnsemble` combines multiple team predictors using Optuna-optimized weights.

In the real Stage 2 sprint this cell receives calibration structures + labels from the organizers and partner model objects from other teams.

In [ ]:
import numpy as np
from pymatgen.core import Structure, Lattice
from eumine_databridge.matfed.federation import FederatedEnsemble

# ── Build calibration set (dummy — organizers provide this at the sprint) ───
cal_structures = []
cal_ef_true = []
cal_bg_true = []

cal_data = [
    # (species, positions, lattice_cubic_a, known_ef, known_bg)
    (["Si","Si"], [[0,0,0],[0.25,0.25,0.25]], 5.43,  -0.54, 1.12),
    (["Na","Cl"], [[0,0,0],[0.5,0.5,0.5]],   5.64,  -3.37, 8.50),
    (["Al","Al","Al","Al"], [[0,0,0],[0,0.5,0.5],[0.5,0,0.5],[0.5,0.5,0]], 4.05, -0.04, 0.00),
    (["Cu","Cu","Cu","Cu"], [[0,0,0],[0,0.5,0.5],[0.5,0,0.5],[0.5,0.5,0]], 3.61, -0.04, 0.00),
    (["Mg","O"], [[0,0,0],[0.5,0.5,0.5]], 4.21, -3.09, 7.83),
]

for species, positions, a, ef, bg in cal_data:
    s = Structure(Lattice.cubic(a), species, positions)
    cal_structures.append(s)
    cal_ef_true.append(ef)
    cal_bg_true.append(bg)

print(f"Calibration set: {len(cal_structures)} structures")

# ── Set up federation ───────────────────────────────────────────────────────
fed = FederatedEnsemble()
fed.add_predictor(predictor, "CataLIST")

# In the sprint, you would also add:
#   fed.add_predictor(partner_model_1, "TeamB")
#   fed.add_predictor(partner_model_2, "TeamC")

# ── Fit on calibration data ─────────────────────────────────────────────────
print("\nFitting federation weights on calibration set...")
result = fed.fit(
    cal_structures=cal_structures,
    cal_ef=cal_ef_true,
    cal_bg=cal_bg_true,
    n_trials=50,  # 50 for demo; use 200 in the real sprint
)

# ── Generate federated predictions on test structures ───────────────────────
test_structures = structures_prog  # reuse Si, NaCl, TiO2 from section 5
test_labels = labels_prog

print("\nFederated predictions on test structures:")
fed_preds = fed.predict(test_structures)

print(f"\n{'Structure':<20} {'EF (eV/atom)':>14} {'BG (eV)':>10} {'Model':<40}")
print("-" * 88)
for label, p in zip(test_labels, fed_preds):
    print(f"{label:<20} {p['formation_energy_per_atom']:>14.4f} {p['band_gap']:>10.4f} {p['model_id']:<40}")

print(f"\n  Calibration score : {result['calibration_score']:.4f} / 40")
print(f"  Federated EF MAE  : {result['federated_mae_ef']:.4f} eV/atom")
print(f"  Federated BG MAE  : {result['federated_mae_bg']:.4f} eV")
print("\n✓ Federation demo complete")

---
## Summary

| Step | Status |
|---|---|
| Dependencies installed | ✓ |
| Model downloaded from HF Hub | ✓ |
| Predictor loaded | ✓ |
| MatFed API compliance | ✓ |
| Live predictions (programmatic + CIF) | ✓ |
| Training overview | ✓ |
| Federation demo | ✓ |

**Contact:** euminecost@gmail.com  
**Repository:** https://github.com/Iman-Peivaste/eumine_databridge_2026  
**Model Hub:** https://huggingface.co/Imanpeivaste/catallist_v4_combined